# Qwen3Guard-Stream: потоковый baseline

Ноутбук запускает один токенный проход Qwen3Guard для каждой трассы, сохраняет его и затем воспроизводит шесть режимов буферизации с двумя политиками безопасности. Логика эксперимента импортируется из Python-пакета; полные чувствительные запросы и ответы не выводятся.

## 1. Конфигурация

`smoke` выбирает 5 безопасных и 5 опасных трасс. `full` использует все 500. Для локальной проверки без NVIDIA можно заменить `cuda` на `cpu`, но показатели времени после этого нельзя напрямую сравнивать с запуском на видеокарте.

In [ ]:
import platform
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from streamguard_bench.experiments import (
    load_boundary_dataset,
    run_experiment,
    select_profile_traces,
)
from streamguard_bench.guards import Qwen3GuardStreamAdapter
from streamguard_bench.metrics import compute_streaming_metrics

PROFILE = "smoke"  # "smoke" или "full"
DEVICE = "cuda"
RESUME = True
SEED = 42
DATASET_REVISION = "c36ddf315be89916c64fd6eb3b5a00ea6d505d1a"
MODEL_REVISION = "419364a715de9840d47b1457982f64ff37f90ed4"

MODES = ["token", "chunk_8", "chunk_16", "chunk_32", "sentence", "full_buffered"]
POLICIES = ["strict", "conservative"]
ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
OUTPUT_DIR = ROOT / "data/interim/qwen3guard_baseline"

## 2. Окружение

Эти сведения нужны для воспроизводимости задержки. Версии модели и токенизатора дополнительно сохраняются рядом с результатами.

In [ ]:
environment = {
    "profile": PROFILE,
    "device": DEVICE,
    "python": platform.python_version(),
    "platform": platform.platform(),
    "output_dir": str(OUTPUT_DIR),
}
pd.Series(environment, name="value")

## 3. Загрузка и выбор трасс

Показываются только идентификаторы, итоговые метки и категории. Текст запросов и ответов остаётся внутри объекта данных.

In [ ]:
dataset = load_boundary_dataset(
    revision=DATASET_REVISION,
    cache_dir=ROOT / "data/cache/huggingface",
)
selected = select_profile_traces(dataset, PROFILE, seed=SEED)
print(f"Выбрано трасс: {len(selected)}")
display(selected[["trace_id", "source_split", "label", "harm_categories"]])

## 4. Загрузка Qwen3Guard

Загружается модель `Qwen/Qwen3Guard-Stream-0.6B`. Она не обучается: используются опубликованные веса.

In [ ]:
guard = Qwen3GuardStreamAdapter(
    "Qwen/Qwen3Guard-Stream-0.6B",
    revision=MODEL_REVISION,
    tokenizer_revision=MODEL_REVISION,
    device=DEVICE,
)
print({
    "model": guard.model_id,
    "model_revision": guard.model_revision,
    "tokenizer_revision": guard.tokenizer_revision,
    "device": guard.device,
})

## 5. Запуск или продолжение

После каждой трассы токенные решения сохраняются в `data/interim`. При повторном запуске с `RESUME = True` завершённые трассы не вычисляются заново.

In [ ]:
run = run_experiment(
    dataset=dataset,
    guard=guard,
    profile=PROFILE,
    modes=MODES,
    policies=POLICIES,
    output_dir=OUTPUT_DIR,
    resume=RESUME,
    seed=SEED,
    dataset_revision=DATASET_REVISION,
)
print(f"Результаты: {run.output_dir}")

## 6. Проверка полноты

Для `smoke` ожидается 120 строк, для `full` — 6000. Ошибка одной трассы представлена во всех 12 комбинациях, поэтому её нельзя незаметно принять за безопасный результат.

In [ ]:
expected_traces = 10 if PROFILE == "smoke" else 500
expected_rows = expected_traces * len(MODES) * len(POLICIES)
assert len(run.results) == expected_rows, (len(run.results), expected_rows)
assert run.results[["trace_id", "mode", "policy"]].duplicated().sum() == 0
print(f"Строк: {len(run.results)} из {expected_rows}; ошибок трасс: {len(run.errors)}")
display(run.errors)

## 7. Базовые метрики

> **Важно:** результаты по 10 трассам проверяют работоспособность конвейера. Они не являются статистической оценкой качества Qwen3Guard. Исследовательские выводы можно делать только после профиля `full`.

In [ ]:
metrics = compute_streaming_metrics(
    run.results,
    group_by=["mode", "policy"],
)
display(metrics)

## 8. Сравнение режимов

Графики строятся только по агрегатам. Полные чувствительные ответы не отображаются.

In [ ]:
plot_data = metrics.melt(
    id_vars=["mode", "policy"],
    value_vars=["false_positive_rate", "false_negative_rate"],
    var_name="metric",
    value_name="value",
)
plt.figure(figsize=(12, 5))
sns.barplot(data=plot_data, x="mode", y="value", hue="metric")
plt.title(f"Ошибки Qwen3Guard: политика {POLICIES[0]} и {POLICIES[1]}")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

## 9. Ошибки без чувствительного текста

In [ ]:
if run.errors.empty:
    print("Все выбранные трассы обработаны успешно.")
else:
    display(run.errors[["trace_id", "error_type", "error"]])

## 10. Передача участнику 4

Для полного эксперимента изменить только `PROFILE = "full"` и выполнить ноутбук на одном зафиксированном устройстве. Профиль обработает все 500 трасс и сформирует 6000 строк. Участник 4 затем расширяет анализ по `source_split`, категориям и позиции появления вреда, строит доверительные интервалы и оформляет итоговые выводы.